# Indian Water Quality Analytics and WQI Prediction using Machine Learning

---

**AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026**

| | |
|---|---|
| **Student Name** | Santosh Kumar Yadav |
| **Programme** | Data Analytics with AI Internship 2026 |
| **Organiser** | AICTE \| IBM SkillsBuild |
| **Project Title** | Indian Water Quality Analytics and WQI Prediction using Machine Learning |

---

> **File naming prefix:** `SANTOSHKUMARYADAV`  
> **Source of truth files:** `PROJECT_PLAN.md`, `DATASET_AUDIT.md`, `data/processed/wqi_scores.csv`

## 2. Project Overview

This project performs a complete end-to-end data analytics workflow on Indian surface water quality monitoring data from the Central Pollution Control Board (CPCB) National Water Quality Monitoring Programme (NWMP).

**What this project does:**

1. Audits and documents data quality issues in the raw CPCB dataset.
2. Preprocesses the dataset: BDL substitution, missing value imputation, anomaly flagging.
3. Computes a Modified Weighted Arithmetic Water Quality Index (WQI) for approximately 160 freshwater station-year observations.
4. Conducts exploratory data analysis across 14 visualisations (V01–V14).
5. Trains Ridge Regression and Random Forest Regressor models to predict WQI scores.
6. Evaluates models using 5-fold GroupKFold cross-validation grouped by monitoring station.

**Dataset:** 194 station-year observations across 17 Indian states, years 2021–2023.  
**After exclusions:** 160 freshwater rows used for WQI computation and machine learning.

## 3. Problem Statement

Surface water quality in India varies significantly by geography, water body type, and time. The CPCB NWMP collects annual Min/Max monitoring data for key parameters across hundreds of stations, but individual measurements are not publicly available — only annual minimum and maximum summaries.

**The core analytical question:**

> Given a partial set of directly measured parameters — temperature, assumed dissolved oxygen, pH, BOD, water body type, state, and year — can a model estimate the broader WQI score that would result from measuring all seven monitoring parameters?

This framing is a **partial-observation scenario**: four of the seven WQI parameters (Conductivity, Nitrate-N, Fecal Coliform, Total Coliform) are deliberately withheld from the model to create a genuine prediction task. The remaining three numeric features (DO, pH, BOD) are shared with the WQI target, introducing partial target reconstruction — this is explicitly disclosed throughout.

## 4. Objectives

1. Perform a structured data quality audit on the raw CPCB monitoring dataset.
2. Implement robust preprocessing: BDL handling, dash/missing handling, Min > Max anomaly flagging, group-median imputation.
3. Compute a Modified Weighted Arithmetic WQI for all freshwater stations.
4. Classify stations into WQI quality categories.
5. Produce 14 EDA visualisations (V01–V11) and 3 ML result charts (V12–V14).
6. Train and evaluate Ridge Regression and Random Forest models using station-stratified GroupKFold cross-validation.
7. Produce an honest, reproducible internship submission with all methodological limitations clearly stated.

## 5. Dataset Description

**Source:** CPCB National Water Quality Monitoring Programme (NWMP)  
**File:** `data/Indian_water_data.csv` (read-only; never modified)

| Property | Value |
|---|---|
| Total rows | 194 |
| Columns | 23 |
| Years | 2021, 2022, 2023 |
| States covered | 17 |
| Unique water body types | 11 |
| Unique monitoring stations (STN codes) | 168 |

**Column structure:** Each row is one monitoring station in one year. Parameters are provided as annual Min and Max values (not individual measurements).

**Parameters in the dataset:**
- Temperature (°C) — Min, Max
- `Dissolved` (assumed DO, mg/L) — Min, Max *(UNVERIFIED column name interpretation — see Section 17)*
- pH — Min, Max
- Conductivity (µS/cm) — Min, Max
- BOD (mg/L) — Min, Max
- Nitrate-N (mg/L) — Min, Max
- Fecal Coliform (MPN/100ml) — Min, Max
- Total Coliform (MPN/100ml) — Min, Max
- `Fecal - Min/Max` (truncated column name, identity unverified — excluded from analysis)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings('ignore')

# Load processed datasets (raw CSV is never loaded directly in the notebook)
wqi = pd.read_csv('data/processed/wqi_scores.csv')
proc = pd.read_csv('data/processed/water_quality_processed.csv')

print(f"WQI dataset shape: {wqi.shape}")
print(f"Processed dataset shape: {proc.shape}")
print()
print("WQI dataset columns:")
print(list(wqi.columns))
print()
print("First 5 rows (key columns):")
print(wqi[['STN_code','Monitoring_Location','Year','State_Name','Water_Body_Type',
           'WQI_applicable','WQI_score','WQI_category']].head(5).to_string())

## 6. Dataset Quality Audit

A full data quality audit is documented in `DATASET_AUDIT.md`. Key findings:

| Issue | Count | Action |
|---|---|---|
| BDL (Below Detection Limit) values | 18 values across 6 columns | Replaced with UNVERIFIED proxy MDL/2; flagged |
| Dash (`-`) values (meaning ambiguous) | Several across measurement columns | Treated as missing (NaN); flagged |
| Min > Max anomalies | 15 rows | Flagged; NOT corrected (source verification required) |
| Missing numeric values | ~7% across measurement columns | Group-median imputation per `Water_Body_Type` |
| Saline water bodies excluded from WQI | 34 rows (MARINE, SEA, BEACH, saline CREEK) | Flagged `WQI_applicable=0` |

**Unverified assumptions:**
- The column `Dissolved - Min/Max` is assumed to be Dissolved Oxygen (mg/L) — **UNVERIFIED** from the data dictionary.
- BDL proxy substitution values use published typical instrument MDLs — **UNVERIFIED** from CPCB lab documentation.
- Column `Fecal - Min/Max` has truncated/unverified name and 32–53% missing values — **excluded entirely**.

In [ ]:
# Water body type distribution
print("Water body type distribution:")
print(wqi['Water_Body_Type'].value_counts().to_string())
print()
print("WQI_applicable distribution:")
print(wqi['WQI_applicable'].value_counts().to_string())
print()
print("Year distribution:")
print(wqi['Year'].value_counts().sort_index().to_string())
print()
print("State distribution (17 states):")
print(wqi['State_Name'].value_counts().to_string())

## 7. Data Preprocessing

Implemented in `backend/preprocess.py` and executed by `run_preprocessing.py`.

All preprocessing is applied to a copy — the raw CSV is never modified.

### Step-by-step pipeline

| Step | Function | Description |
|---|---|---|
| 1 | `load_raw_data()` | Read raw CSV as strings; verify 23 columns |
| 2 | `rename_columns()` | Positional mapping to clean internal names; `Dissolved` → `DO_assumed` (UNVERIFIED) |
| 3 | `handle_bdl()` | `BDL` → proxy MDL/2; add `*_BDL_flag` columns |
| 4 | `handle_dash()` | `-` → NaN; add `*_dash_flag` columns |
| 5 | `flag_minmax_anomalies()` | Flag `Min > Max` rows; do NOT correct |
| 6 | `parse_numeric_columns()` | Convert measurement strings to float \| None |
| 7 | `impute_missing()` | Group-median imputation per `Water_Body_Type`; add `*_imputed_flag` |
| 8 | `flag_wqi_applicability()` | Mark saline rows `WQI_applicable=0` |

### BDL substitution values (UNVERIFIED proxies)

| Parameter | Substitution Value | Source assumption |
|---|---|---|
| DO_assumed | 0.05 mg/L | Half of ~0.1 mg/L typical DO meter MDL |
| BOD | 0.25 mg/L | Half of ~0.5 mg/L typical BOD5 MDL |
| Conductivity | 0.5 µS/cm | Half of ~1.0 µS/cm typical MDL |
| Nitrate-N | 0.05 mg/L | Half of ~0.1 mg/L typical IC MDL |
| Fecal/Total Coliform | 1.0 MPN/100ml | MPN lower bound ~2 / 2 |

> ⚠ These substitution values are **UNVERIFIED proxies** based on published typical instrument MDLs. Actual CPCB laboratory detection limits were not available.

In [ ]:
# Preprocessing audit log summary
audit = pd.read_csv('data/processed/preprocessing_audit_log.csv')
print(f"Preprocessing audit log: {len(audit)} transformation events")
print()
print("Steps performed:")
print(audit.groupby('step')['rows_affected'].sum().to_string())
print()
print("Full audit log (first 10 entries):")
print(audit[['step','column','action','rows_affected']].head(10).to_string())

In [ ]:
# Imputation summary for ML features
imp_flag_cols = [c for c in wqi.columns if c.endswith('_imputed_flag')]
wqi_valid = wqi[wqi['WQI_applicable'] == 1]

print("Imputation flags in WQI-applicable rows (160 freshwater rows):")
for col in imp_flag_cols:
    n = wqi_valid[col].sum()
    if n > 0:
        print(f"  {col}: {n} rows imputed")

rows_with_any_imputed = (wqi_valid[imp_flag_cols].sum(axis=1) > 0).sum()
print(f"\nRows with at least one imputed ML feature: {rows_with_any_imputed} of {len(wqi_valid)}")
print("(~9.4% of the ML dataset; upstream imputation limitation — see Section 17)")

## 8. Modified Weighted Arithmetic WQI Methodology

> **DISCLOSURE:** This is a **Modified Weighted Arithmetic WQI** following the formula structure of Brown et al. (1970), with adaptations for the available Indian water-quality parameters and standards (BIS IS:10500-2012). It is **not identical to any single published WQI**.

### Formula

$$WQI = \frac{\sum(Q_i \times W_i)}{\sum W_i}$$

where:

$$W_i = \frac{K}{S_i}, \quad K = \frac{1}{\sum(1/S_i)}$$

$$Q_i = \frac{C_i - V_{\text{ideal},i}}{S_i - V_{\text{ideal},i}} \times 100$$

### Annual representative concentration

$$C_i = \frac{\text{Min}_i + \text{Max}_i}{2}$$

> **DISCLOSURE:** This is a **range-midpoint approximation**, NOT the arithmetic mean of individual sample measurements taken during the year. Actual annual means are not available in this dataset.

### Parameter standards and weights

| Parameter | Standard $S_i$ | Ideal $V_i$ | Sub-index formula |
|---|---|---|---|
| DO (assumed) | 6.0 mg/L | 14.6 mg/L | $Q_{DO} = (C_{DO} / S_{DO}) \times 100$ (beneficial; inverted) |
| pH | 8.5 | 7.0 | $Q_{pH} = |C_{pH} - 7.0| / 1.5 \times 100$ (absolute deviation) |
| BOD | 3.0 mg/L | 0 | Standard formula |
| Conductivity | 300 µS/cm | 0 | Standard formula |
| Nitrate-N | 10.2 mg/L | 0 | Standard formula |
| Fecal Coliform | 500 MPN/100ml | 0 | Standard formula |
| Total Coliform | 5000 MPN/100ml | 0 | Standard formula |

Standards: BIS IS:10500-2012 (drinking water) and CPCB DBU criteria.

### WQI Classification

| WQI Score | Category |
|---|---|
| 0 – 25 | Excellent |
| 25 – 50 | Good |
| 50 – 75 | Poor |
| 75 – 100 | Very Poor |
| > 100 | Unsuitable for Drinking |

Classification thresholds are project-specific (based on Tyagi et al. 2013 structure). Exact threshold values have not been independently verified against the original paper.

## 9. WQI Results

In [ ]:
wqi_valid = wqi[wqi['WQI_applicable'] == 1].copy()
print(f"Freshwater rows with computed WQI: {len(wqi_valid)}")
print()
print("WQI score descriptive statistics:")
print(wqi_valid['WQI_score'].describe().round(2).to_string())
print()
print("WQI category distribution:")
cat_counts = wqi_valid['WQI_category'].value_counts()
for cat, count in cat_counts.items():
    pct = 100 * count / len(wqi_valid)
    print(f"  {cat:<30} {count:>3} rows  ({pct:.1f}%)")

In [ ]:
print("Mean WQI score by state (ascending = better quality):")
state_wqi = wqi_valid.groupby('State_Name')['WQI_score'].mean().sort_values()
for state, score in state_wqi.items():
    print(f"  {state:<25} {score:.2f}")

print()
print("Mean WQI score by water body type:")
wb_wqi = wqi_valid.groupby('Water_Body_Type')['WQI_score'].mean().sort_values()
for wb, score in wb_wqi.items():
    print(f"  {wb:<40} {score:.2f}")

## 10. Exploratory Data Analysis

Visualisations V01–V11 were generated by `stage4_eda_charts.py` and saved to `report_images/`.  
They are displayed here for reference — the original PNG files are the authoritative output.

> **Note:** Do not regenerate these charts. The existing saved files are the Stage 4 output.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V01_wqi_distribution.png'))
ax.axis('off')
ax.set_title('Figure 1. Distribution of computed WQI scores (160 freshwater rows)', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V02_wqi_category_counts.png'))
ax.axis('off')
ax.set_title('Figure 2. WQI category counts', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V03_wqi_by_state.png'))
ax.axis('off')
ax.set_title('Figure 3. WQI score distribution by state', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V04_wqi_by_water_body_type.png'))
ax.axis('off')
ax.set_title('Figure 4. WQI score distribution by water body type', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V05_parameter_correlation.png'))
ax.axis('off')
ax.set_title('Figure 5. Parameter correlation heatmap', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V06_do_vs_bod.png'))
ax.axis('off')
ax.set_title('Figure 6. Assumed DO vs BOD scatter plot', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V07_fc_vs_tc.png'))
ax.axis('off')
ax.set_title('Figure 7. Fecal Coliform vs Total Coliform scatter plot', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V08_missing_values.png'))
ax.axis('off')
ax.set_title('Figure 8. Missing values heatmap', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V09_temporal_trend.png'))
ax.axis('off')
ax.set_title('Figure 9. Temporal WQI trend (multi-year stations)', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V10_top_10_polluted_stations.png'))
ax.axis('off')
ax.set_title('Figure 10. Top 10 most polluted stations by WQI score', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V11_top_10_cleanest_stations.png'))
ax.axis('off')
ax.set_title('Figure 11. Top 10 cleanest stations by WQI score', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

### Key EDA Findings

- **WQI distribution:** Right-skewed; most stations fall in the "Poor" (50–75) band. Mean WQI = 68.0, median = 61.2.
- **By state:** Uttar Pradesh, Haryana, and Goa show the lowest mean WQI (better quality). Odisha, Tamil Nadu, and Maharashtra show the highest mean WQI.
- **By water body type:** Sewage Treatment Plants (STPs) show markedly elevated WQI scores (mean 114). Water Treatment Plants (raw water intake) show the lowest mean WQI (44).
- **DO vs BOD:** Strong negative correlation; stations with high BOD tend to have low assumed DO — consistent with organic pollution degrading oxygen levels.
- **FC vs TC:** Strong positive correlation across several orders of magnitude, as expected.
- **Temporal:** Limited multi-year coverage (26 stations appear in 2 years). No strong consistent trend visible with this sample size.

## 11. Machine Learning Methodology

### 11.1 Problem framing

**Target:** `WQI_score` — the computed Modified WQI score (continuous, range ~41 to ~142 in the freshwater subset)

**Rows used:** 160 freshwater rows (WQI_applicable = 1, WQI_score not null)

**Feature set (Option A — Final Implementation Scope):**

| Feature | Type | Notes |
|---|---|---|
| `Temp_mean` | Numeric | (Temp_Min + Temp_Max) / 2; **not** a WQI component |
| `DO_assumed_mean` | Numeric | UNVERIFIED assumed DO; **is** a WQI component — partial leakage |
| `pH_mean` | Numeric | **is** a WQI component — partial leakage |
| `BOD_mean` | Numeric | **is** a WQI component — partial leakage |
| `Year` | Numeric (ordinal) | Temporal signal |
| `State_Name` | Categorical | 17 states; one-hot encoded |
| `Water_Body_Type` | Categorical | 7 types (freshwater subset); one-hot encoded |

**Deliberately withheld (to avoid full WQI reconstruction):**  
`Conductivity_mean`, `NitrateN_mean`, `FC_mean`, `TC_mean`

> ⚠ **PARTIAL TARGET LEAKAGE DISCLOSURE:** `DO_assumed_mean`, `pH_mean`, and `BOD_mean` are direct components of the WQI formula. Including them as features means the model can partially reconstruct the WQI from these inputs. This is **not independent external prediction of water quality**; it is a partial-formula reconstruction task with four parameters withheld. This must be considered when interpreting model performance metrics.

### 11.2 Excluded identifiers

`STN_code` and `Monitoring_Location` are excluded from all feature sets. These are station identifiers that would cause near-perfect memorisation if included.

### 11.3 Pipeline design

All preprocessing is performed **inside** an sklearn `Pipeline`:

```
Numeric features:  SimpleImputer(strategy='median') → StandardScaler
Categorical:       SimpleImputer(strategy='most_frequent') → OneHotEncoder(handle_unknown='ignore')
```

Imputation statistics are fitted **only on the training fold**, preventing validation data from influencing preprocessing.

### 11.4 Upstream imputation caveat

Stage 3 preprocessing applied group-median imputation over the full 194-row dataset before any fold split. For the ~15 rows (9.4%) with imputed ML feature values, residual leakage from the Stage 3 imputation cannot be fully eliminated. This is an explicit limitation of the dataset and pipeline as designed.

## 12. Ridge Regression

**Model:** `sklearn.linear_model.Ridge(alpha=1.0)`

Ridge Regression is a regularised linear model (L2 penalty). It serves as the linear baseline for this project.

**Configuration:**

| Parameter | Value |
|---|---|
| alpha (regularisation strength) | 1.0 |
| Fit intercept | True (default) |
| Solver | auto (default) |

### Cross-validation results (5-fold GroupKFold)

| Fold | Train n | Val n | RMSE | R² |
|---|---|---|---|---|
| 1 | 128 | 32 | 10.46 | 0.558 |
| 2 | 128 | 32 | 11.67 | 0.630 |
| 3 | 128 | 32 | 13.56 | 0.487 |
| 4 | 128 | 32 | 21.48 | 0.387 |
| 5 | 128 | 32 | 50.58 | −3.043 |
| **Mean ± SD** | | | **21.55 ± 16.79** | **−0.196 ± 1.594** |

Ridge Regression shows high variance across folds (Fold 5 RMSE = 50.58 vs Fold 1 RMSE = 10.46). The negative mean R² indicates the model fails to explain variance on average across all folds, largely driven by Fold 5.

## 13. Random Forest Regressor

**Model:** `sklearn.ensemble.RandomForestRegressor`

Random Forest is an ensemble of decision trees that handles non-linearity and is robust to outliers.

**Configuration:**

| Parameter | Value |
|---|---|
| n_estimators | 200 |
| min_samples_leaf | 3 |
| max_depth | None (full trees) |
| random_state | 42 |

### Cross-validation results (5-fold GroupKFold)

| Fold | Train n | Val n | RMSE | R² |
|---|---|---|---|---|
| 1 | 128 | 32 | 3.44 | 0.952 |
| 2 | 128 | 32 | 3.25 | 0.971 |
| 3 | 128 | 32 | 4.11 | 0.953 |
| 4 | 128 | 32 | 5.82 | 0.955 |
| 5 | 128 | 32 | 6.89 | 0.925 |
| **Mean ± SD** | | | **4.70 ± 1.59** | **0.951 ± 0.017** |

Under the selected 5-fold GroupKFold evaluation, Random Forest produced lower mean RMSE and higher mean R² than Ridge Regression. The high R² should be interpreted in the context of partial target leakage — see Section 17.

## 14. GroupKFold Cross-Validation Strategy

**Validator:** `sklearn.model_selection.GroupKFold(n_splits=5)`  
**Groups:** `STN_code` (monitoring station identifier)

### Why station-grouped validation?

26 monitoring stations appear in two different years (2021 and 2022/2023). A random row split could place Year 1 of a station in the training set and Year 2 in the validation set. Since water quality at the same station is likely correlated across years, this would allow the model to implicitly memorise station characteristics — inflating validation scores.

By assigning **all rows for a given station to the same fold**, GroupKFold ensures that the validation set contains only stations the model has never seen during training.

### Why no separate held-out test set?

With 160 rows and 150 unique stations, a 15% held-out test set would contain only ~24 rows / ~22 stations. This is too small to produce statistically reliable evaluation estimates. The 5-fold GroupKFold CV result (reporting mean ± std across 5 folds) provides a more stable estimate of generalisation performance than a single small held-out set.

The final models (`model/best_model.pkl` and `model/ridge_model.pkl`) are retrained on all 160 rows after cross-validation.

In [ ]:
# Display fold structure
import json
with open('model/model_metadata.json') as f:
    meta = json.load(f)

print("Validation strategy:", meta['validation'])
print("Groups:", "STN_code")
print("n_rows:", meta['n_rows'])
print("n_stations:", meta['n_stations'])
print()
print("Fold sizes (from RF results):")
for fold_r in meta['models']['random_forest']['fold_results']:
    print(f"  Fold {fold_r['fold']}: train={fold_r['train_n']}, val={fold_r['val_n']}")

## 15. Model Evaluation

### Summary comparison table

In [ ]:
print("=" * 70)
print("Model Comparison — 5-fold GroupKFold CV Results")
print("=" * 70)
print(f"{'Model':<42} {'Mean RMSE':>12}  {'Mean R2':>10}")
print("-" * 70)

models_display = [
    ("Ridge Regression (alpha=1.0)",
     meta['models']['ridge_regression']),
    ("Random Forest (n=200, min_leaf=3)",
     meta['models']['random_forest']),
]

for name, m in models_display:
    print(f"{name:<42} {m['mean_rmse']:.4f} +/- {m['std_rmse']:.4f}  "
          f"{m['mean_r2']:.4f} +/- {m['std_r2']:.4f}")

print()
print("Under the selected 5-fold GroupKFold evaluation, Random Forest")
print("produced lower mean RMSE and higher mean R2 than Ridge Regression.")
print()
print("IMPORTANT: The high Random Forest R2 (0.951) is partly attributable")
print("to partial target reconstruction -- BOD, DO, and pH are WQI components.")
print("BOD alone accounts for ~96% of the Random Forest feature importance.")

## 16. Feature Importance and Model Visualisations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V12_feature_importance.png'))
ax.axis('off')
ax.set_title('Figure 12. Random Forest feature importance (mean decrease in impurity)', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

> **Interpretation:** `BOD_mean` dominates the feature importance (~95.8%), which is consistent with the partial target leakage: BOD is a major WQI component with standard value 3.0 mg/L. The model is largely recovering the relationship between BOD and WQI that is built into the WQI formula. `DO_assumed_mean` and `pH_mean` contribute smaller shares. Temperature, year, state, and water body type have minimal importance.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V13_actual_vs_predicted.png'))
ax.axis('off')
ax.set_title('Figure 13. Actual vs predicted WQI scores (out-of-fold)', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(mpimg.imread(r'report_images/V14_residuals.png'))
ax.axis('off')
ax.set_title('Figure 14. Residual plot (out-of-fold predictions)', fontsize=11, pad=8)
plt.tight_layout()
plt.show()

> **Residual interpretation:** Random Forest residuals are distributed symmetrically around zero with no clear systematic pattern. Ridge Regression shows systematic under/over-prediction for high WQI values, consistent with the model's inability to capture non-linear relationships between BOD and WQI in the upper range.

## 17. Limitations and Scientific Disclosures

All of the following limitations and unverified assumptions are explicitly disclosed here and carried through the project documentation.

---

### 17.1 Unverified Dissolved Oxygen Interpretation

The column `Dissolved - Min/Max` in the raw dataset is interpreted as Dissolved Oxygen (mg/L). This interpretation is supported by:
- Value range 0.3–13.6 mg/L, consistent with DO in aquatic environments.
- High values at cold mountain rivers; low values at heavily polluted stations (STPs, drains).
- CPCB NWMP standard parameter set includes DO, and `Dissolved` fills the DO slot.

However, this has **not been confirmed** against the CPCB data dictionary or source documentation. All derived columns are labelled `DO_assumed` throughout the project.

**Impact if wrong:** All WQI computations using DO would be invalid.

---

### 17.2 Modified/Custom WQI Index

The WQI computed in this project is a **Modified Weighted Arithmetic WQI** following the formula structure of Brown et al. (1970). Key adaptations are project-specific:
- Standard values ($S_i$) selected from BIS IS:10500-2012.
- DO sub-index uses an inverted beneficial-parameter formula.
- pH sub-index uses absolute deviation from neutral (7.0).
- Classification thresholds follow the Tyagi et al. (2013) structure but exact threshold verification was not possible.
- Annual concentrations use (Min + Max) / 2 as a range-midpoint approximation.

This WQI is **not identical to any single published index** and should not be described as such.

---

### 17.3 Range-Midpoint Approximation

Annual representative parameter values are computed as:

$$C_i = (\text{Min}_i + \text{Max}_i) / 2$$

This is a **range-midpoint approximation**. It equals the true arithmetic mean only if the Min and Max are equally distributed around the mean, which is unlikely for environmental monitoring data. Actual annual means are not available in this dataset.

---

### 17.4 Unverified BDL Substitution Values

BDL (Below Detection Limit) values were replaced with UNVERIFIED proxy values (MDL/2 based on published typical instrument MDLs). The actual CPCB laboratory detection limits were not available. These are flagged throughout the processed dataset.

---

### 17.5 Partial Target Reconstruction

The ML features `DO_assumed_mean`, `pH_mean`, and `BOD_mean` are also **direct components of the WQI formula**. Including them as features allows the model to partially reconstruct the WQI from these inputs.

This is **not independent external prediction of water quality**. It is more accurately described as: "given three of the seven WQI parameters, can the model approximate the WQI that would result from measuring all seven?"

The dominant importance of `BOD_mean` (~95.8%) in the Random Forest confirms that the model has largely learned the BOD → WQI relationship encoded in the formula.

`Conductivity_mean`, `NitrateN_mean`, `FC_mean`, and `TC_mean` are deliberately withheld to prevent full formula reconstruction.

---

### 17.6 Upstream Imputation Limitation

Stage 3 preprocessing (`run_preprocessing.py`) applied group-median imputation over the full 194-row dataset **before** any fold split. For approximately 15 rows (9.4%) with imputed ML feature values, the imputation statistics were computed using information from rows that would later become validation folds.

Stage 5 (`stage5_ml.py`) re-applies in-fold median imputation inside the sklearn Pipeline to mitigate this. However, it **cannot fully eliminate** the residual leakage introduced by the Stage 3 upstream imputation.

This pipeline is **not completely leakage-free** for the ~9.4% of imputed rows.

---

### 17.7 Small Dataset

After excluding 34 saline rows, 160 freshwater rows remain for WQI computation and ML. The dataset has 150 unique monitoring stations.

- Cross-validation results with a 160-row dataset should be treated as indicative, not definitive.
- No separate held-out test set was created (a ~25-row test set would be unreliable).
- Complex models (deep networks, large boosted ensembles) were not used — they would overfit at this scale.

---

### 17.8 Saline Water Body Exclusion

34 rows are excluded from the drinking-water-oriented WQI analysis:
- MARINE: 10 rows
- BEACH: 19 rows
- SEA: 4 rows
- CREEK (saline, Conductivity > 5,000 µS/cm): 1 row

BIS IS:10500-2012 drinking water standards are inapplicable to saline water.

## 18. Conclusion

This project completed a structured end-to-end data analytics workflow on Indian CPCB water quality monitoring data:

1. **Data audit and preprocessing** were performed transparently with full documentation of all transformations, assumptions, and their limitations in `DATASET_AUDIT.md`, `PROJECT_PLAN.md`, and the preprocessing audit log.

2. **WQI computation** produced scores for 160 freshwater station-year observations. The majority (65.6%) fall in the "Poor" category (WQI 50–75). States such as Uttar Pradesh and Haryana showed lower (better) mean WQI, while Odisha, Tamil Nadu, and Maharashtra showed higher mean WQI scores. Sewage Treatment Plants consistently showed the worst water quality.

3. **EDA (V01–V11)** revealed expected patterns: strong DO–BOD negative correlation, wide state-level variation, and the dominance of BOD as a predictor of water quality status.

4. **Machine Learning (V12–V14):** Under 5-fold GroupKFold cross-validation (groups = STN_code), Random Forest produced mean RMSE = 4.70 ± 1.59 and mean R² = 0.951 ± 0.017, compared to Ridge Regression's mean RMSE = 21.55 ± 16.79 and mean R² = −0.196 ± 1.594. The high Random Forest performance is substantially driven by partial target reconstruction via the BOD feature.

**Key honest conclusion:** The Random Forest model is effective at approximating the WQI formula from partial inputs, but this is a reconstruction task rather than genuinely independent water quality prediction. A model that predicts water quality without any WQI-component features as inputs remains a goal for future work with richer datasets.

---

## References

- Brown, R.M., McClelland, N.I., Deininger, R.A., & Tozer, R.G. (1970). A water quality index — do we dare? *Water and Sewage Works*, 117, 339–343.
- Bureau of Indian Standards (2012). *IS:10500 — Drinking Water Specification (Second Revision).* New Delhi: BIS.
- Central Pollution Control Board (CPCB). National Water Quality Monitoring Programme (NWMP). New Delhi: CPCB.
- Tyagi, S., Sharma, B., Singh, P., & Dobhal, R. (2013). Water quality assessment in terms of water quality index. *American Journal of Water Resources*, 1(3), 34–38. DOI: 10.12691/ajwr-1-3-3.

---

*Submitted by Santosh Kumar Yadav as part of the AICTE | IBM SkillsBuild Data Analytics with AI Internship 2026.*